# Library management tool

This lab starts a tool to manage a library's loans, with SQLAlchemy's ORM on a SQLite database: model the books, the members and their loans, import data from CSV files, query the database, export the loans as CSV, then analyze them with pandas. It assumes the ORM part of the "Getting started with SQLAlchemy" lab.

The steps build on each other: each solution defines what the next ones use. If you're stuck, run the solution before moving on.

## Setup

The next three cells write the library's data, one table per CSV file: the books (`books.csv`), the members (`members.csv`), and the dated loans (`loans.csv`), some of them returned (they have a `returned` date). The one after creates the engine, on the `library.db` file, and the models' `Base` class.

In [ ]:
%%writefile books.csv
title,author
Vingt Mille Lieues sous les mers,Jules Verne
Voyage au centre de la Terre,Jules Verne
De la Terre à la Lune,Jules Verne
Le Tour du monde en quatre-vingts jours,Jules Verne
Michel Strogoff,Jules Verne
The Hound of the Baskervilles,Arthur Conan Doyle
"Arsène Lupin, gentleman-cambrioleur",Maurice Leblanc

In [ ]:
%%writefile members.csv
name
John Doe
Jane Doe
Sherlock Holmes
Arsène Lupin

In [ ]:
%%writefile loans.csv
book,member,date,returned
Voyage au centre de la Terre,Jane Doe,2026-01-03,2026-01-24
Le Tour du monde en quatre-vingts jours,Sherlock Holmes,2026-01-05,2026-01-19
Michel Strogoff,Sherlock Holmes,2026-01-07,
"Arsène Lupin, gentleman-cambrioleur",Jane Doe,2026-01-20,2026-02-01
The Hound of the Baskervilles,Sherlock Holmes,2026-02-02,
Voyage au centre de la Terre,John Doe,2026-02-10,
Le Tour du monde en quatre-vingts jours,Sherlock Holmes,2026-02-20,2026-03-06

In [ ]:
from sqlalchemy import create_engine, select
from sqlalchemy.orm import DeclarativeBase, Session

engine = create_engine("sqlite:///library.db")


class Base(DeclarativeBase):
  pass

## 1. Modeling books, members and loans

A member borrows several books, and a book is borrowed by several members over time: it's a many-to-many relationship. Each loan also carries information of its own, its date and its return date: it becomes a class in its own right, an *association object*. Describe, on the `Base` provided:

- `Book` (table `books`): `id`; `title`, unique; `author`; `loans`, the list of its loans;
- `Member` (table `members`): `id`; `name`, unique; `loans`;
- `Loan` (table `loans`): `id`; `id_book` and `id_member`, foreign keys; `loan_date`, a date; `return_date`, an optional date (an ongoing loan has none); `book` and `member`, the object on each side.

Drop the existing tables, then create these ones. Why does a loan have its own `id`, rather than a primary key made of `id_book` and `id_member`?

**Useful documentation**

- [Association object](https://docs.sqlalchemy.org/en/21/orm/basic_relationships.html#association-object)

In [ ]:
# Your code here

### Additional hints

A date is annotated `Mapped[date]` (`from datetime import date`), an optional column `Mapped[date | None]`, and `mapped_column(unique=True)` forbids duplicates. There are two relationships, each with its two sides, which name each other with `back_populates`: `Book.loans` and `Loan.book`, `Member.loans` and `Loan.member`.

```python
from datetime import date

from sqlalchemy import ForeignKey
from sqlalchemy.orm import Mapped, mapped_column, relationship


class Book(Base):
  __tablename__ = "books"
  id: Mapped[int] = mapped_column(primary_key=True)
  title: Mapped[str] = mapped_column(unique=True)
  author: Mapped[str]
  loans: Mapped[list["Loan"]] = relationship(back_populates="book")


class Member(Base):
  __tablename__ = "members"
  ...


class Loan(Base):
  __tablename__ = "loans"
  id: Mapped[int] = mapped_column(primary_key=True)
  id_book: Mapped[int] = mapped_column(ForeignKey(...))
  ...
  book: Mapped[Book] = relationship(back_populates=...)
  ...


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)
```

### Solution

In [ ]:
from datetime import date

from sqlalchemy import ForeignKey
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship


class Base(DeclarativeBase):
  pass


class Book(Base):
  __tablename__ = "books"
  id: Mapped[int] = mapped_column(primary_key=True)
  title: Mapped[str] = mapped_column(unique=True)
  author: Mapped[str]
  loans: Mapped[list["Loan"]] = relationship(back_populates="book")


class Member(Base):
  __tablename__ = "members"
  id: Mapped[int] = mapped_column(primary_key=True)
  name: Mapped[str] = mapped_column(unique=True)
  loans: Mapped[list["Loan"]] = relationship(back_populates="member")


class Loan(Base):
  __tablename__ = "loans"
  id: Mapped[int] = mapped_column(primary_key=True)
  id_book: Mapped[int] = mapped_column(ForeignKey("books.id"))
  id_member: Mapped[int] = mapped_column(ForeignKey("members.id"))
  loan_date: Mapped[date]
  return_date: Mapped[date | None]
  book: Mapped[Book] = relationship(back_populates="loans")
  member: Mapped[Member] = relationship(back_populates="loans")


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# The solution starts from a new Base: declaring a Book class a second time on
# the same Base (if you already did above) would raise an error.
#
# A member can borrow the same book several times (in the data, Sherlock Holmes
# and Le Tour du monde en quatre-vingts jours): the pair (id_book, id_member)
# doesn't identify a loan, so it can't be the primary key. Hence an id of each
# loan's own.

## 2. Importing the CSV files

Complete the function `import_csv(directory, session)`, which reads the files `books.csv`, `members.csv` and `loans.csv` of the directory `directory`, and adds to the session a `Book` per book, a `Member` per member and a `Loan` per loan. Loans refer to books and members by their title and name: keep the objects you create in dictionaries to find them. Then call it on the current directory (`Path(".")`) in a session, commit, and print the number of books, members and loans in the database.

**Useful documentation**

- [`csv.DictReader`](https://docs.python.org/3/library/csv.html#csv.DictReader)
- [`date.fromisoformat`](https://docs.python.org/3/library/datetime.html#datetime.date.fromisoformat)

In [ ]:
from pathlib import Path


def import_csv(directory: Path, session: Session) -> None:
  """Add the books, members and loans of the CSV files of directory to the session."""
  ...

In [ ]:
# Your code here

### Additional hints

`csv.DictReader(f)` reads a CSV file row by row, each row as a dictionary whose keys are the header's column names; open the file with `newline=""`, as the `csv` module requires. The title holding a comma is quoted in the file: `csv` removes the quotes. Every value is a string: `date.fromisoformat` converts the dates, and `returned` is an empty string for an ongoing loan. Adding the books, the members and the loans to the session is enough: the relationships link them. You should get 7 books, 4 members and 7 loans.

```python
import csv


def read_csv(path: Path) -> list[dict[str, str]]:
  with open(path, newline="", encoding="utf-8") as f:
    return list(csv.DictReader(f))


def import_csv(directory: Path, session: Session) -> None:
  books = {
    row["title"]: Book(title=row["title"], author=row["author"])
    for row in read_csv(directory / "books.csv")
  }
  members = {
    row["name"]: Member(name=row["name"]) for row in read_csv(directory / "members.csv")
  }
  session.add_all(books.values())
  session.add_all(members.values())
  for row in read_csv(directory / "loans.csv"):
    returned = row["returned"]
    session.add(
      Loan(
        book=...,
        member=...,
        loan_date=...,
        return_date=...,
      )
    )


with Session(engine) as session:
  import_csv(Path("."), session)
  session.commit()
  for model in (Book, Member, Loan):
    print(model.__name__, len(session.scalars(select(model)).all()))
```

### Solution

In [ ]:
import csv


def read_csv(path: Path) -> list[dict[str, str]]:
  with open(path, newline="", encoding="utf-8") as f:
    return list(csv.DictReader(f))


def import_csv(directory: Path, session: Session) -> None:
  books = {
    row["title"]: Book(title=row["title"], author=row["author"])
    for row in read_csv(directory / "books.csv")
  }
  members = {
    row["name"]: Member(name=row["name"]) for row in read_csv(directory / "members.csv")
  }
  session.add_all(books.values())
  session.add_all(members.values())
  for row in read_csv(directory / "loans.csv"):
    returned = row["returned"]
    session.add(
      Loan(
        book=books[row["book"]],
        member=members[row["member"]],
        loan_date=date.fromisoformat(row["date"]),
        return_date=date.fromisoformat(returned) if returned else None,
      )
    )


with Session(engine) as session:
  import_csv(Path("."), session)
  session.commit()
  for model in (Book, Member, Loan):
    print(model.__name__, len(session.scalars(select(model)).all()))

## 3. Querying the database

Each query of this part is written with `select`, and run in a session with `session.scalars(...)`.

### The books Sherlock Holmes borrowed

Print the title of the books Sherlock Holmes borrowed, with each loan's date, sorted by date.

In [ ]:
# Your code here

#### Additional hints

Start from `select(Loan)`, join `Loan.member`, filter on `Member.name` and sort by `Loan.loan_date` with `order_by`. Each loan gives its book through `loan.book`. You should get four loans, two of them of the same book, from January 5 to February 20, 2026.

```python
with Session(engine) as session:
  query = select(Loan).join(...).where(...).order_by(...)
  for loan in session.scalars(query):
    print(loan.loan_date, loan.book.title)
```

#### Solution

In [ ]:
with Session(engine) as session:
  query = (
    select(Loan)
    .join(Loan.member)
    .where(Member.name == "Sherlock Holmes")
    .order_by(Loan.loan_date)
  )
  for loan in session.scalars(query):
    print(loan.loan_date, loan.book.title)

### The ongoing loans

Print the ongoing loans, those without a return date, by loan date: the date, the book's title and the member's name.

In [ ]:
# Your code here

#### Additional hints

In SQL, a missing value is tested with `IS NULL`, which SQLAlchemy writes `Loan.return_date.is_(None)`. You should get 3 loans.

```python
with Session(engine) as session:
  query = select(Loan).where(...).order_by(Loan.loan_date)
  for loan in session.scalars(query):
    print(...)
```

#### Solution

In [ ]:
with Session(engine) as session:
  ongoing = Loan.return_date.is_(None)
  query = select(Loan).where(ongoing).order_by(Loan.loan_date)
  for loan in session.scalars(query):
    print(loan.loan_date, loan.book.title, loan.member.name)

### The available books

Print, in alphabetical order, the title of the available books: those with no ongoing loan.

**Useful documentation**

- [`any`](https://docs.sqlalchemy.org/en/21/orm/internals.html#sqlalchemy.orm.RelationshipProperty.Comparator.any)

In [ ]:
# Your code here

#### Additional hints

`Book.loans.any(condition)` is true for a book with at least one loan meeting the condition (in SQL, an `EXISTS` with a `WHERE`), and `~` negates it. The "ongoing" condition is the previous exercise's. Beware: a returned book may have been borrowed again. You should get 4 books.

```python
with Session(engine) as session:
  ongoing = Loan.return_date.is_(None)
  query = select(Book).where(...).order_by(Book.title)
  for book in session.scalars(query):
    print(book.title)
```

#### Solution

In [ ]:
with Session(engine) as session:
  ongoing = Loan.return_date.is_(None)
  query = select(Book).where(~Book.loans.any(ongoing)).order_by(Book.title)
  for book in session.scalars(query):
    print(book.title)

# Voyage au centre de la Terre isn't there: returned by Jane Doe, it was borrowed
# again, by John Doe. A book is available if it has no ongoing loan, not if it
# has a returned loan.

## 4. Exporting the loans as CSV

Complete the function `export_csv(path, session)`, which writes every loan, by loan date, to the CSV file `path`: one row per loan, with the columns `book`, `member`, `loan_date` and `return_date` (empty for an ongoing loan). Call it to write `export.csv`, then print the file's contents.

**Useful documentation**

- [`csv.DictWriter`](https://docs.python.org/3/library/csv.html#csv.DictWriter)
- [`date.isoformat`](https://docs.python.org/3/library/datetime.html#datetime.date.isoformat)

In [ ]:
def export_csv(path: str, session: Session) -> None:
  """Write the loans of the database to the CSV file at path."""
  ...

In [ ]:
# Your code here

### Additional hints

`csv.DictWriter(f, fieldnames=[...])` writes dictionaries: `writeheader()` writes the header, then `writerow(...)` each row. Open the file with `newline=""`, as the `csv` module requires. `isoformat()` writes a date as `2026-01-03`. The file should have 8 lines: the header and 7 loans.

```python
import csv


def export_csv(path: str, session: Session) -> None:
  with open(path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
      f, fieldnames=["book", "member", "loan_date", "return_date"]
    )
    writer.writeheader()
    for loan in session.scalars(select(Loan).order_by(Loan.loan_date)):
      writer.writerow(
        {
          "book": loan.book.title,
          "member": ...,
          "loan_date": ...,
          "return_date": ...,
        }
      )


with Session(engine) as session:
  export_csv("export.csv", session)

with open("export.csv", encoding="utf-8") as f:
  print(f.read())
```

### Solution

In [ ]:
import csv


def export_csv(path: str, session: Session) -> None:
  with open(path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
      f, fieldnames=["book", "member", "loan_date", "return_date"]
    )
    writer.writeheader()
    for loan in session.scalars(select(Loan).order_by(Loan.loan_date)):
      returned = loan.return_date
      writer.writerow(
        {
          "book": loan.book.title,
          "member": loan.member.name,
          "loan_date": loan.loan_date.isoformat(),
          "return_date": returned.isoformat() if returned else "",
        }
      )


with Session(engine) as session:
  export_csv("export.csv", session)

with open("export.csv", encoding="utf-8") as f:
  print(f.read())

## 5. Analyzing the loans with pandas

With `pd.read_sql`, load into a `DataFrame` `loans` the result of a `select` query giving, for each loan, the member's name (column `member`), the book's title (column `book`) and the loan date. Then print the number of loans of each member, from the largest to the smallest.

**Useful documentation**

- [`ColumnElement.label`](https://docs.sqlalchemy.org/en/21/core/sqlelement.html#sqlalchemy.sql.expression.ColumnElement.label)
- [`Series.value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)

In [ ]:
# Your code here

### Additional hints

`pd.read_sql(query, engine)` runs the query and returns a `DataFrame`, one column per item of the `select`, and `.label("member")` names a column. Start from the loan, `select_from(Loan)`, to join its two relationships. `value_counts()` counts a column's values, from the most frequent to the least. You should get Sherlock Holmes 4, Jane Doe 2 and John Doe 1.

```python
import pandas as pd

query = (
  select(Member.name.label("member"), ..., Loan.loan_date)
  .select_from(Loan)
  .join(...)
  .join(...)
)
loans = pd.read_sql(query, engine)
print(...)
```

### Solution

In [ ]:
import pandas as pd

query = (
  select(Member.name.label("member"), Book.title.label("book"), Loan.loan_date)
  .select_from(Loan)
  .join(Loan.member)
  .join(Loan.book)
)
loans = pd.read_sql(query, engine)
print(loans["member"].value_counts())